In [9]:
import sounddevice as sd
import numpy as np
from piper.voice import PiperVoice

class TextToSpeech:
    """
    Class to handle text-to-speech using Piper.
    Only speaks text that ends with two spaces.
    """
    def __init__(self, model_path: str, config_path: str):
        self.voice = PiperVoice.load(model_path, config_path)
        self.default_sample_rate = 16000

    def _synthesize_audio(self, text: str) -> (bytes, int):
        """
        Generate audio bytes from text using Piper.
        Returns combined bytes and the sample rate.
        """
        audio_chunks = []
        sample_rate = self.default_sample_rate

        for chunk in self.voice.synthesize(text):
            audio_chunks.append(chunk.audio_int16_bytes)
            sample_rate = chunk.sample_rate

        combined_audio = b"".join(audio_chunks)
        return combined_audio, sample_rate

    def speak(self, text: str):
        """
        Speak the text if it ends with two spaces.
        """
        if not text.endswith("  "):
            return  # Ignore text without the ending condition

        audio_bytes, sample_rate = self._synthesize_audio(text)
        audio_array = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / 32768.0
        sd.play(audio_array, samplerate=sample_rate)
        sd.wait()


if __name__ == "__main__":
    # Initialize TTS system
    tts = TextToSpeech("en_US-amy-medium.onnx", "en_US-amy-medium.onnx.json")

    # Example text
    my_text = "Hello to Model letters!  "

    # Speak text
    tts.speak(my_text)
